In [23]:
import pandas as pd
import numpy as np

print("=" * 70)
print("STEP 4 최종 병합 데이터 검증")
print("=" * 70)

# 검증 대상 (메모리의 train_df, 없으면 파일에서 로드)
df = pd.read_csv("../../data/processed/1_merge/step_4-4_weather.csv", low_memory=False)

# 원본 재로드 (대조용)
raw_train   = pd.read_csv("../../data/raw/hanwoo_train.csv")
raw_lineage = pd.read_csv("../../data/raw/hanwoo_lineage.csv", low_memory=False)
raw_death   = pd.read_csv("../../data/raw/hanwoo_death.csv")
raw_area    = pd.read_csv("../../data/raw/hanwoo_area.csv")

def check(name, condition):
    print(f"  [{'PASS' if condition else 'FAIL'}] {name}")
    return condition

# =====================================================
# 섹션 1. 기본 구조
# =====================================================
print("\n[섹션 1] 기본 구조")
check("행 수 = 원본 train(2,408,699)", len(df) == len(raw_train))
check("열 수 = 44", df.shape[1] == 44)
check("CATTLE_NO 중복 없음", df["CATTLE_NO"].duplicated().sum() == 0)
check("CATTLE_NO 결측 없음", df["CATTLE_NO"].isnull().sum() == 0)
check("CATTLE_NO 집합이 원본과 동일",
      set(df["CATTLE_NO"]) == set(raw_train["CATTLE_NO"]))
print(f"  컬럼 목록: {df.columns.tolist()}")

# =====================================================
# 섹션 2. 병합 정합성
# =====================================================
print("\n[섹션 2] 병합 정합성")

# --- train 원본 컬럼 유지 확인 ---
train_cols = ["sido", "sigungu", "eupmyeondong", "stn", "ABATT_DATE",
              "JUDGE_DATE", "JUDGE_SEX", "WEIGHT", "BACKFAT", "REA",
              "WINDEX", "WGRADE", "INSFAT", "YUKSAK", "FATSAK", "TISSUE",
              "GROWTH", "COST_AMT", "AGE", "BIRTH_YMD", "CATTLE_NO",
              "FARM_UNIQUE_NO", "LAST_GRADE"]
check("train 원본 23개 컬럼 전부 존재",
      all(c in df.columns for c in train_cols))

# train 원본 값 변조 여부 (샘플 10개 대조)
sample_idx = raw_train.sample(10, random_state=42)["CATTLE_NO"].tolist()
for col in ["WEIGHT", "AGE", "LAST_GRADE"]:
    for cno in sample_idx:
        orig = raw_train[raw_train["CATTLE_NO"] == cno][col].iloc[0]
        merged = df[df["CATTLE_NO"] == cno][col].iloc[0]
        if str(orig) != str(merged):
            check(f"train 원본값 변조 없음 ({col})", False)
            break
    else:
        check(f"train 원본값 변조 없음 ({col})", True)

# --- lineage ---
lineage_cattle = set(raw_lineage["CATTLE_NO"])
expected_lineage_null = (~df["CATTLE_NO"].isin(lineage_cattle)).sum()
actual_lineage_null = df["KPN_NO"].isnull().sum()
check(f"lineage 결측({actual_lineage_null:,}) = 원본 미등록({expected_lineage_null:,})",
      actual_lineage_null == expected_lineage_null)

# 7개 컬럼 모두 존재
lineage_cols = ["KPN_NO", "FATHER_CATTLE_NO", "MOTHER_ANIMAL_NO",
                "F_GMOTHER_ANIMAL_NO", "F_GFATHER_CATTLE_NO",
                "M_GMOTHER_ANIMAL_NO", "M_GFATHER_CATTLE_NO"]
check("lineage 7개 컬럼 존재", all(c in df.columns for c in lineage_cols))

# 결측 패턴: 7개 컬럼 결측이 동일한 행인지
lineage_null_counts = df[lineage_cols].isnull().sum(axis=1)
check("lineage 7개 컬럼 결측 패턴 동일 (전부 있거나 전부 없거나)",
      lineage_null_counts.isin([0, 7]).all())

# 샘플 값 정확성
sample_cattle = df[df["KPN_NO"].notnull()]["CATTLE_NO"].iloc[0]
df_kpn  = df[df["CATTLE_NO"] == sample_cattle]["KPN_NO"].iloc[0]
raw_kpn = raw_lineage[raw_lineage["CATTLE_NO"] == sample_cattle]["KPN_NO"].iloc[0]
check("lineage 값 정확성 (샘플 KPN_NO 일치)", df_kpn == raw_kpn)

# --- death ---
death_farms = set(raw_death["FARM_UNIQUE_NO"])
expected_death_null = (~df["FARM_UNIQUE_NO"].isin(death_farms)).sum()
actual_death_null = df["death_count"].isnull().sum()
check(f"death 결측({actual_death_null:,}) = death_df 없는 농장({expected_death_null:,})",
      actual_death_null == expected_death_null)
check("death_count 컬럼 존재", "death_count" in df.columns)

# 샘플 값 정확성
death_summary_check = raw_death.groupby("FARM_UNIQUE_NO").size()
sample_farm = df[df["death_count"].notnull()]["FARM_UNIQUE_NO"].iloc[0]
df_dc  = df[df["FARM_UNIQUE_NO"] == sample_farm]["death_count"].iloc[0]
raw_dc = death_summary_check[sample_farm]
check("death_count 값 정확성 (샘플 농장 일치)", df_dc == raw_dc)

# death_count 양수 여부 (있는 값은 0 이상)
check("death_count 값 >= 0", (df["death_count"].dropna() >= 0).all())

# --- area ---
area_farms = set(raw_area["FARM_UNIQUE_NO"])
expected_area_null = (~df["FARM_UNIQUE_NO"].isin(area_farms)).sum()
check("area 4개 컬럼 존재",
      all(c in df.columns for c in ["C2023","C2024","C2025","AREA"]))
check(f"area 농장없음 소속 행 = {expected_area_null:,} (C2023 결측은 이 이상)",
      df["C2023"].isnull().sum() >= expected_area_null)

# area 샘플 값 정확성
area_summary = raw_area.copy()
for c in ["C2023","C2024","C2025","AREA"]:
    area_summary[c] = area_summary[c].replace(-99, np.nan)
area_summary = area_summary.groupby("FARM_UNIQUE_NO")[["C2023","C2024","C2025","AREA"]].sum()
sample_area_farm = df[df["C2023"].notnull()]["FARM_UNIQUE_NO"].iloc[0]
df_c2023  = df[df["FARM_UNIQUE_NO"] == sample_area_farm]["C2023"].iloc[0]
raw_c2023 = area_summary.loc[sample_area_farm, "C2023"] if sample_area_farm in area_summary.index else np.nan
check("area C2023 값 정확성 (샘플 농장 일치)",
      abs(df_c2023 - raw_c2023) < 0.01 if not np.isnan(raw_c2023) else True)

# --- weather ---
weather_cols = ["days_total", "days_양호", "days_주의", "days_경고",
                "days_위험", "days_폐사", "rn_day_mean", "ws_davg_mean", "ta_min_mean"]
check("weather 9개 컬럼 존재", all(c in df.columns for c in weather_cols))
check("days_total 결측 없음", df["days_total"].isnull().sum() == 0)
check("days_total > 0 (모든 소가 최소 1일 이상)",
      (df["days_total"] > 0).all())
grade_sum = df[["days_양호","days_주의","days_경고","days_위험","days_폐사"]].sum(axis=1)
check("등급별 일수 합 = 총일수 (전체)",
      (grade_sum == df["days_total"]).all())
check("days 등급별 >= 0",
      (df[["days_양호","days_주의","days_경고","days_위험","days_폐사"]] >= 0).all().all())
check("rn_day_mean >= 0", (df["rn_day_mean"] >= 0).all())
check("ws_davg_mean >= 0", (df["ws_davg_mean"] >= 0).all())

# =====================================================
# 섹션 3. BIRTH_YMD 무결성 (과거 오류 재발 방지)
# =====================================================
print("\n[섹션 3] BIRTH_YMD 무결성")

df["BIRTH_YMD"]  = pd.to_datetime(df["BIRTH_YMD"], errors="coerce")
df["ABATT_DATE"] = pd.to_datetime(df["ABATT_DATE"], errors="coerce")
df["JUDGE_DATE"] = pd.to_datetime(df["JUDGE_DATE"], errors="coerce")

check("BIRTH_YMD 결측 없음", df["BIRTH_YMD"].isnull().sum() == 0)
check("BIRTH_YMD에 1970-01-01 없음 (과거 DB 삽입 오류)",
      (df["BIRTH_YMD"] == "1970-01-01").sum() == 0)
check("BIRTH_YMD 최솟값 >= 1996", df["BIRTH_YMD"].min().year >= 1996)
check("BIRTH_YMD <= ABATT_DATE (출생일이 도축일보다 이전)",
      (df["BIRTH_YMD"] <= df["ABATT_DATE"]).all())
check("ABATT_DATE 결측 없음", df["ABATT_DATE"].isnull().sum() == 0)
check("ABATT_DATE 범위 2023~2025",
      df["ABATT_DATE"].dt.year.between(2023, 2025).all())

# =====================================================
# 섹션 4. 데이터 타입
# =====================================================
print("\n[섹션 4] 데이터 타입")

check("BIRTH_YMD datetime", pd.api.types.is_datetime64_any_dtype(df["BIRTH_YMD"]))
check("ABATT_DATE datetime", pd.api.types.is_datetime64_any_dtype(df["ABATT_DATE"]))
check("JUDGE_DATE datetime", pd.api.types.is_datetime64_any_dtype(df["JUDGE_DATE"]))

num_cols = ["WEIGHT", "BACKFAT", "REA", "AGE", "COST_AMT",
            "WINDEX", "INSFAT", "YUKSAK", "FATSAK", "TISSUE", "GROWTH",
            "death_count", "C2023", "C2024", "C2025", "AREA",
            "days_total", "days_양호", "days_주의", "days_경고",
            "days_위험", "days_폐사", "rn_day_mean", "ws_davg_mean", "ta_min_mean"]
for c in num_cols:
    check(f"{c} 수치형", pd.api.types.is_numeric_dtype(df[c]))

cat_cols = ["sido", "JUDGE_SEX", "LAST_GRADE", "CATTLE_NO", "FARM_UNIQUE_NO"]
for c in cat_cols:
    check(f"{c} 문자형",
          df[c].dtype == object or pd.api.types.is_string_dtype(df[c]))

# =====================================================
# 섹션 5. 결측치 현황
# =====================================================
print("\n[섹션 5] 결측치 현황")

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({"결측수": missing, "결측률(%)": missing_pct})
print(summary[summary["결측수"] > 0].sort_values("결측률(%)", ascending=False).to_string())

# 전체 NaN 컬럼 없는지
all_nan_cols = [c for c in df.columns if df[c].isnull().all()]
check(f"전체 NaN인 컬럼 없음 (발견: {all_nan_cols})", len(all_nan_cols) == 0)

# 육질 컬럼 결측이 등외에서만 발생하는지
quality_cols = ["BACKFAT", "REA", "WINDEX", "INSFAT", "YUKSAK", "FATSAK", "TISSUE"]
quality_null_grades = df[df["BACKFAT"].isnull()]["LAST_GRADE"].unique()
check("육질컬럼 결측이 전부 등외", set(quality_null_grades) <= {"등외"})

# COST_AMT 결측이 특정 등급에 편중되지 않는지
cost_null_grades = df[df["COST_AMT"].isnull()]["LAST_GRADE"].value_counts()
check("COST_AMT 결측이 전 등급에 분산 (편중 없음)",
      cost_null_grades.min() > 0)

# =====================================================
# 섹션 6. 컬럼 무결성
# =====================================================
print("\n[섹션 6] 컬럼 무결성")

# 상수 컬럼 확인 (days_폐사 제외 - THI max < 98이라 정상)
const_cols = [c for c in df.columns
              if df[c].nunique(dropna=True) <= 1 and c != "days_폐사"]
check(f"상수 컬럼 없음 (days_폐사 제외, 발견: {const_cols})",
      len(const_cols) == 0)
check("days_폐사 전부 0 (THI max 97.65 < 98, 정상)",
      (df["days_폐사"] == 0).all())

# WEIGHT, AGE 기본 범위
check("WEIGHT > 0", (df["WEIGHT"].dropna() > 0).all())
check("AGE > 0", (df["AGE"].dropna() > 0).all())

# =====================================================
# 섹션 7. 종속변수
# =====================================================
print("\n[섹션 7] 종속변수 LAST_GRADE")

print("\n등급 분포:")
print(df["LAST_GRADE"].value_counts().to_string())
check("LAST_GRADE 결측 없음", df["LAST_GRADE"].isnull().sum() == 0)
check("등외 포함 16개 등급", df["LAST_GRADE"].nunique() == 16)

# 클래스 불균형 확인
max_ratio = df["LAST_GRADE"].value_counts().max() / len(df) * 100
min_ratio = df["LAST_GRADE"].value_counts().min() / len(df) * 100
print(f"\n  최다 클래스 비율: {max_ratio:.2f}%")
print(f"  최소 클래스 비율: {min_ratio:.2f}%")
print(f"  불균형 비율: {max_ratio/min_ratio:.1f}배")

print("\n" + "=" * 70)
print("검증 완료")
print("=" * 70)

STEP 4 최종 병합 데이터 검증

[섹션 1] 기본 구조
  [PASS] 행 수 = 원본 train(2,408,699)
  [PASS] 열 수 = 44
  [PASS] CATTLE_NO 중복 없음
  [PASS] CATTLE_NO 결측 없음
  [PASS] CATTLE_NO 집합이 원본과 동일
  컬럼 목록: ['sido', 'sigungu', 'eupmyeondong', 'stn', 'ABATT_DATE', 'JUDGE_DATE', 'JUDGE_SEX', 'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'WGRADE', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH', 'COST_AMT', 'AGE', 'BIRTH_YMD', 'CATTLE_NO', 'FARM_UNIQUE_NO', 'LAST_GRADE', 'death_count', 'KPN_NO', 'FATHER_CATTLE_NO', 'MOTHER_ANIMAL_NO', 'F_GMOTHER_ANIMAL_NO', 'F_GFATHER_CATTLE_NO', 'M_GMOTHER_ANIMAL_NO', 'M_GFATHER_CATTLE_NO', 'C2023', 'C2024', 'C2025', 'AREA', 'days_total', 'days_양호', 'days_주의', 'days_경고', 'days_위험', 'days_폐사', 'rn_day_mean', 'ws_davg_mean', 'ta_min_mean']

[섹션 2] 병합 정합성
  [PASS] train 원본 23개 컬럼 전부 존재
  [PASS] train 원본값 변조 없음 (WEIGHT)
  [PASS] train 원본값 변조 없음 (AGE)
  [PASS] train 원본값 변조 없음 (LAST_GRADE)
  [PASS] lineage 결측(922,653) = 원본 미등록(922,653)
  [PASS] lineage 7개 컬럼 존재
  [PASS] lineage 7개 컬럼 결측 패턴 동일 (전부 

In [24]:
print("=== 수치형 -99 현황 ===")
for col in df.select_dtypes(include=[np.number]).columns:
    cnt = (df[col] == -99).sum()
    if cnt > 0:
        print(f"{col}: {cnt:,}개")

print("\n=== 문자형 '-99' 현황 ===")
for col in df.columns:
    try:
        cnt = (df[col] == "-99").sum()
        if cnt > 0:
            print(f"{col}: {cnt:,}개")
    except:
        pass

=== 수치형 -99 현황 ===
BACKFAT: 4,567개
REA: 5,188개
WINDEX: 5,252개
INSFAT: 5,331개
YUKSAK: 5,269개
FATSAK: 4,743개
TISSUE: 5,320개
GROWTH: 9개
COST_AMT: 885,003개

=== 문자형 '-99' 현황 ===
KPN_NO: 3,570개


In [25]:
print("\n[섹션 0] -99 처리")

# 처리 전 현황
print("처리 전 -99 현황:")
for col in df.select_dtypes(include=[np.number]).columns:
    cnt = (df[col] == -99).sum()
    if cnt > 0:
        print(f"  {col}: {cnt:,}개")
for col in df.columns:
    try:
        cnt = (df[col] == "-99").sum()
        if cnt > 0:
            print(f"  {col} (문자): {cnt:,}개")
    except:
        pass

# 수치형 -99 → NaN
num_minus99_cols = ["BACKFAT", "REA", "WINDEX", "INSFAT",
                    "YUKSAK", "FATSAK", "TISSUE", "GROWTH", "COST_AMT"]
for col in num_minus99_cols:
    df[col] = df[col].replace(-99, np.nan)

# 문자형 '-99' → NaN
df["KPN_NO"] = df["KPN_NO"].replace("-99", np.nan)

# 처리 후 확인
print("\n처리 후 -99 잔여:")
has_minus99 = False
for col in df.select_dtypes(include=[np.number]).columns:
    cnt = (df[col] == -99).sum()
    if cnt > 0:
        print(f"  {col}: {cnt:,}개")
        has_minus99 = True
for col in df.columns:
    try:
        cnt = (df[col] == "-99").sum()
        if cnt > 0:
            print(f"  {col} (문자): {cnt:,}개")
            has_minus99 = True
    except:
        pass
check("-99 전부 처리 완료", not has_minus99)


[섹션 0] -99 처리
처리 전 -99 현황:
  BACKFAT: 4,567개
  REA: 5,188개
  WINDEX: 5,252개
  INSFAT: 5,331개
  YUKSAK: 5,269개
  FATSAK: 4,743개
  TISSUE: 5,320개
  GROWTH: 9개
  COST_AMT: 885,003개
  KPN_NO (문자): 3,570개

처리 후 -99 잔여:
  [PASS] -99 전부 처리 완료


True

In [29]:
# =====================================================
# 섹션 1. 기본 구조
# =====================================================
print("\n[섹션 1] 기본 구조")

check("행 수 = 원본 train(2,408,699)", len(df) == len(raw_train))
check("열 수 = 44", df.shape[1] == 44)
check("CATTLE_NO 중복 없음", df["CATTLE_NO"].duplicated().sum() == 0)
check("CATTLE_NO 결측 없음", df["CATTLE_NO"].isnull().sum() == 0)
check("CATTLE_NO 집합이 원본과 동일",
      set(df["CATTLE_NO"]) == set(raw_train["CATTLE_NO"]))
print(f"  컬럼 목록: {df.columns.tolist()}")

# =====================================================
# 섹션 2. 병합 정합성
# =====================================================
print("\n[섹션 2] 병합 정합성")

# train
train_cols = ["sido", "sigungu", "eupmyeondong", "stn", "ABATT_DATE",
              "JUDGE_DATE", "JUDGE_SEX", "WEIGHT", "BACKFAT", "REA",
              "WINDEX", "WGRADE", "INSFAT", "YUKSAK", "FATSAK", "TISSUE",
              "GROWTH", "COST_AMT", "AGE", "BIRTH_YMD", "CATTLE_NO",
              "FARM_UNIQUE_NO", "LAST_GRADE"]
check("train 원본 23개 컬럼 전부 존재",
      all(c in df.columns for c in train_cols))

sample_idx = raw_train.sample(10, random_state=42)["CATTLE_NO"].tolist()
for col in ["WEIGHT", "AGE", "LAST_GRADE"]:
    for cno in sample_idx:
        orig   = raw_train[raw_train["CATTLE_NO"] == cno][col].iloc[0]
        merged = df[df["CATTLE_NO"] == cno][col].iloc[0]
        if str(orig) != str(merged):
            check(f"train 원본값 변조 없음 ({col})", False)
            break
    else:
        check(f"train 원본값 변조 없음 ({col})", True)

# lineage
lineage_cattle = set(raw_lineage["CATTLE_NO"])
not_in_lineage = (~df["CATTLE_NO"].isin(lineage_cattle)).sum()
expected_kpn_null = not_in_lineage + 3570  # '-99' 변환 3,570개 포함
actual_kpn_null   = df["KPN_NO"].isnull().sum()
check(f"KPN_NO 결측({actual_kpn_null:,}) = 미등록({not_in_lineage:,}) + '-99'변환(3,570)",
      actual_kpn_null == expected_kpn_null)

lineage_cols = ["KPN_NO", "FATHER_CATTLE_NO", "MOTHER_ANIMAL_NO",
                "F_GMOTHER_ANIMAL_NO", "F_GFATHER_CATTLE_NO",
                "M_GMOTHER_ANIMAL_NO", "M_GFATHER_CATTLE_NO"]
check("lineage 7개 컬럼 존재", all(c in df.columns for c in lineage_cols))

lineage_null_counts = df[lineage_cols].isnull().sum(axis=1)
check("lineage 결측 패턴 정상 (0개 또는 1개 또는 7개)",
      lineage_null_counts.isin([0, 1, 7]).all())

sample_cattle = df[df["KPN_NO"].notnull()]["CATTLE_NO"].iloc[0]
df_kpn  = df[df["CATTLE_NO"] == sample_cattle]["KPN_NO"].iloc[0]
raw_kpn = raw_lineage[raw_lineage["CATTLE_NO"] == sample_cattle]["KPN_NO"].iloc[0]
check("lineage 값 정확성 (샘플 KPN_NO 일치)", df_kpn == raw_kpn)

# death
death_farms = set(raw_death["FARM_UNIQUE_NO"])
expected_death_null = (~df["FARM_UNIQUE_NO"].isin(death_farms)).sum()
actual_death_null   = df["death_count"].isnull().sum()
check(f"death 결측({actual_death_null:,}) = death_df 없는 농장({expected_death_null:,})",
      actual_death_null == expected_death_null)
check("death_count 컬럼 존재", "death_count" in df.columns)

death_summary_check = raw_death.groupby("FARM_UNIQUE_NO").size()
sample_farm = df[df["death_count"].notnull()]["FARM_UNIQUE_NO"].iloc[0]
df_dc  = df[df["FARM_UNIQUE_NO"] == sample_farm]["death_count"].iloc[0]
raw_dc = death_summary_check[sample_farm]
check("death_count 값 정확성 (샘플 농장 일치)", df_dc == raw_dc)
check("death_count 값 >= 0", (df["death_count"].dropna() >= 0).all())

# area
area_farms = set(raw_area["FARM_UNIQUE_NO"])
expected_area_null = (~df["FARM_UNIQUE_NO"].isin(area_farms)).sum()
check("area 4개 컬럼 존재",
      all(c in df.columns for c in ["C2023","C2024","C2025","AREA"]))
check(f"area 농장없음 소속 행 = {expected_area_null:,} (C2023 결측은 이 이상)",
      df["C2023"].isnull().sum() >= expected_area_null)

area_summary = raw_area.copy()
for c in ["C2023","C2024","C2025","AREA"]:
    area_summary[c] = area_summary[c].replace(-99, np.nan)
area_summary = area_summary.groupby("FARM_UNIQUE_NO")[["C2023","C2024","C2025","AREA"]].sum()
sample_area_farm = df[df["C2023"].notnull()]["FARM_UNIQUE_NO"].iloc[0]
df_c2023  = df[df["FARM_UNIQUE_NO"] == sample_area_farm]["C2023"].iloc[0]
raw_c2023 = area_summary.loc[sample_area_farm, "C2023"] if sample_area_farm in area_summary.index else np.nan
check("area C2023 값 정확성 (샘플 농장 일치)",
      abs(df_c2023 - raw_c2023) < 0.01 if not np.isnan(raw_c2023) else True)

# weather
weather_cols = ["days_total", "days_양호", "days_주의", "days_경고",
                "days_위험", "days_폐사", "rn_day_mean", "ws_davg_mean", "ta_min_mean"]
check("weather 9개 컬럼 존재", all(c in df.columns for c in weather_cols))
check("days_total 결측 없음", df["days_total"].isnull().sum() == 0)
check("days_total > 0", (df["days_total"] > 0).all())
grade_sum = df[["days_양호","days_주의","days_경고","days_위험","days_폐사"]].sum(axis=1)
check("등급별 일수 합 = 총일수", (grade_sum == df["days_total"]).all())
check("days 등급별 >= 0",
      (df[["days_양호","days_주의","days_경고","days_위험","days_폐사"]] >= 0).all().all())
check("rn_day_mean >= 0", (df["rn_day_mean"] >= 0).all())
check("ws_davg_mean >= 0", (df["ws_davg_mean"] >= 0).all())

# =====================================================
# 섹션 3. BIRTH_YMD 무결성
# =====================================================
print("\n[섹션 3] BIRTH_YMD 무결성")

df["BIRTH_YMD"]  = pd.to_datetime(df["BIRTH_YMD"], errors="coerce")
df["ABATT_DATE"] = pd.to_datetime(df["ABATT_DATE"], errors="coerce")
df["JUDGE_DATE"] = pd.to_datetime(df["JUDGE_DATE"], errors="coerce")

check("BIRTH_YMD 결측 없음", df["BIRTH_YMD"].isnull().sum() == 0)
check("BIRTH_YMD에 1970-01-01 없음", (df["BIRTH_YMD"] == "1970-01-01").sum() == 0)
check("BIRTH_YMD 최솟값 >= 1996", df["BIRTH_YMD"].min().year >= 1996)
check("BIRTH_YMD <= ABATT_DATE", (df["BIRTH_YMD"] <= df["ABATT_DATE"]).all())
check("ABATT_DATE 결측 없음", df["ABATT_DATE"].isnull().sum() == 0)
check("ABATT_DATE 범위 2023~2025",
      df["ABATT_DATE"].dt.year.between(2023, 2025).all())

# =====================================================
# 섹션 4. 데이터 타입
# =====================================================
print("\n[섹션 4] 데이터 타입")

check("BIRTH_YMD datetime", pd.api.types.is_datetime64_any_dtype(df["BIRTH_YMD"]))
check("ABATT_DATE datetime", pd.api.types.is_datetime64_any_dtype(df["ABATT_DATE"]))
check("JUDGE_DATE datetime", pd.api.types.is_datetime64_any_dtype(df["JUDGE_DATE"]))

num_cols = ["WEIGHT", "BACKFAT", "REA", "AGE", "COST_AMT",
            "WINDEX", "INSFAT", "YUKSAK", "FATSAK", "TISSUE", "GROWTH",
            "death_count", "C2023", "C2024", "C2025", "AREA",
            "days_total", "days_양호", "days_주의", "days_경고",
            "days_위험", "days_폐사", "rn_day_mean", "ws_davg_mean", "ta_min_mean"]
for c in num_cols:
    check(f"{c} 수치형", pd.api.types.is_numeric_dtype(df[c]))

cat_cols = ["sido", "JUDGE_SEX", "LAST_GRADE", "CATTLE_NO", "FARM_UNIQUE_NO"]
for c in cat_cols:
    check(f"{c} 문자형",
          df[c].dtype == object or pd.api.types.is_string_dtype(df[c]))

# =====================================================
# 섹션 5. 결측치 현황
# =====================================================
print("\n[섹션 5] 결측치 현황")

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({"결측수": missing, "결측률(%)": missing_pct})
print(summary[summary["결측수"] > 0].sort_values("결측률(%)", ascending=False).to_string())

all_nan_cols = [c for c in df.columns if df[c].isnull().all()]
check(f"전체 NaN인 컬럼 없음 (발견: {all_nan_cols})", len(all_nan_cols) == 0)

quality_cols = ["BACKFAT", "REA", "WINDEX", "INSFAT", "YUKSAK", "FATSAK", "TISSUE"]
quality_null_grades = df[df["BACKFAT"].isnull()]["LAST_GRADE"].unique()
check("육질컬럼 결측이 전부 등외", set(quality_null_grades) <= {"등외"})

cost_null_cnt = df["COST_AMT"].isnull().sum()
check(f"COST_AMT 결측 수 확인 ({cost_null_cnt:,}개)", cost_null_cnt > 0)

# =====================================================
# 섹션 6. 컬럼 무결성
# =====================================================
print("\n[섹션 6] 컬럼 무결성")

const_cols = [c for c in df.columns
              if df[c].nunique(dropna=True) <= 1 and c != "days_폐사"]
check(f"상수 컬럼 없음 (days_폐사 제외, 발견: {const_cols})", len(const_cols) == 0)
check("days_폐사 전부 0 (THI max 97.65 < 98, 정상)", (df["days_폐사"] == 0).all())
check("WEIGHT > 0", (df["WEIGHT"].dropna() > 0).all())
check("AGE > 0", (df["AGE"].dropna() > 0).all())

# =====================================================
# 섹션 7. 종속변수
# =====================================================
print("\n[섹션 7] 종속변수 LAST_GRADE")

print("\n등급 분포:")
print(df["LAST_GRADE"].value_counts().to_string())
check("LAST_GRADE 결측 없음", df["LAST_GRADE"].isnull().sum() == 0)
check("등외 포함 16개 등급", df["LAST_GRADE"].nunique() == 16)

max_ratio = df["LAST_GRADE"].value_counts().max() / len(df) * 100
min_ratio = df["LAST_GRADE"].value_counts().min() / len(df) * 100
print(f"\n  최다 클래스 비율: {max_ratio:.2f}%")
print(f"  최소 클래스 비율: {min_ratio:.2f}%")
print(f"  불균형 비율: {max_ratio/min_ratio:.1f}배")


[섹션 1] 기본 구조
  [PASS] 행 수 = 원본 train(2,408,699)
  [PASS] 열 수 = 44
  [PASS] CATTLE_NO 중복 없음
  [PASS] CATTLE_NO 결측 없음
  [PASS] CATTLE_NO 집합이 원본과 동일
  컬럼 목록: ['sido', 'sigungu', 'eupmyeondong', 'stn', 'ABATT_DATE', 'JUDGE_DATE', 'JUDGE_SEX', 'WEIGHT', 'BACKFAT', 'REA', 'WINDEX', 'WGRADE', 'INSFAT', 'YUKSAK', 'FATSAK', 'TISSUE', 'GROWTH', 'COST_AMT', 'AGE', 'BIRTH_YMD', 'CATTLE_NO', 'FARM_UNIQUE_NO', 'LAST_GRADE', 'death_count', 'KPN_NO', 'FATHER_CATTLE_NO', 'MOTHER_ANIMAL_NO', 'F_GMOTHER_ANIMAL_NO', 'F_GFATHER_CATTLE_NO', 'M_GMOTHER_ANIMAL_NO', 'M_GFATHER_CATTLE_NO', 'C2023', 'C2024', 'C2025', 'AREA', 'days_total', 'days_양호', 'days_주의', 'days_경고', 'days_위험', 'days_폐사', 'rn_day_mean', 'ws_davg_mean', 'ta_min_mean']

[섹션 2] 병합 정합성
  [PASS] train 원본 23개 컬럼 전부 존재
  [PASS] train 원본값 변조 없음 (WEIGHT)
  [PASS] train 원본값 변조 없음 (AGE)
  [PASS] train 원본값 변조 없음 (LAST_GRADE)
  [PASS] KPN_NO 결측(926,223) = 미등록(922,653) + '-99'변환(3,570)
  [PASS] lineage 7개 컬럼 존재
  [PASS] lineage 결측 패턴 정상 (0개 또는 1개 또는 7개)


In [30]:
df.to_csv("../../data/processed/1_merge/step5_merge.csv",
          index=False, encoding="utf-8-sig")
print(f"저장 완료: {df.shape}")

저장 완료: (2408699, 44)
